# Agent-Driven Feature Research System for Financial Time-Series Classification

Author: Candidate

## 0. Result Index
- 数据概况输出：Section 2
- 特征诊断输出：Section 6
- 数据泄漏检查输出：Section 5
- 训练/评估指标输出：Section 9
- ROC/混淆矩阵可视化：Section 10
- Top 50 特征分析：Section 8/10

## 1. Setup & Imports

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_io import load_dataset, chronological_split, detect_leakage, ensure_columns_exist
from src.utils.config import PipelineConfig
from src.features.diagnose import run_feature_diagnosis
from src.features.clean import build_cleaning_plan, apply_cleaning
from src.features.evaluate import add_redundancy_penalty, compute_final_scores
from src.features.select import select_top_features
from src.modeling.validate import train_and_compare

## 2. Data Loading & Overview

In [ ]:
cfg = PipelineConfig()
df = load_dataset(cfg.data_path)
df.shape, df.head()

## 3. Label Screening & Task Definition

In [ ]:
split = chronological_split(df)
train_df, valid_df, test_df = split['train'], split['valid'], split['test']

def label_screening(train_df, label_cols):
    rows = []
    for y in label_cols:
        if y not in train_df.columns:
            continue
        s = train_df[y]
        vals = s.dropna().unique()
        if len(vals) != 2:
            continue
        miss = s.isna().mean()
        bal = min((s == vals[0]).mean(), (s == vals[1]).mean())
        rows.append((y, (1-miss)*bal, miss, bal))
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows

label_rank = label_screening(train_df, cfg.label_columns)
target_col = label_rank[0][0]
target_col, label_rank[:5]

## 4. Agent System Design
Planner -> Diagnosis -> Cleaning -> Selection -> Executor

## 5. Data Leakage Checks

In [ ]:
leakage = detect_leakage(split)
leakage

## 6. Feature Diagnosis

In [ ]:
feature_cols = ensure_columns_exist(train_df, cfg.feature_columns)
diagnosis_df = run_feature_diagnosis(train_df, feature_cols, target_col)
diagnosis_df.head()

## 7. Feature Cleaning

In [ ]:
cleaning_df = build_cleaning_plan(diagnosis_df)
clean_train, clean_valid, clean_test = apply_cleaning(train_df, valid_df, test_df, cleaning_df, target_col)
cleaning_df.head()

## 8. Feature Evaluation & Selection

In [ ]:
usable_features = [c for c in feature_cols if c in clean_train.columns]
redundancy = add_redundancy_penalty(clean_train, usable_features, corr_threshold=cfg.corr_threshold)
score_df = compute_final_scores(diagnosis_df[diagnosis_df['feature'].isin(usable_features)], redundancy)
top50_df = select_top_features(score_df, top_k=cfg.top_k, corr_threshold=cfg.corr_threshold, data_df=clean_train)
top50_df.head()

## 9. Downstream Classification Validation

In [ ]:
import random
random.seed(cfg.random_seed)
feature_sets = {
    'raw_all': feature_cols,
    'cleaned_all': usable_features,
    'agent_top50': top50_df['feature'].tolist(),
    'random_50': random.sample(usable_features, k=min(cfg.top_k, len(usable_features))),
}
metrics_df = train_and_compare(clean_train, clean_valid, clean_test, target_col, feature_sets, cfg.random_seed)
metrics_df

## 10. Visualizations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=diagnosis_df.sort_values('missing_rate', ascending=False).head(30), x='missing_rate', y='feature', ax=ax[0])
ax[0].set_title('Top 30 Missing Ratio')
sns.barplot(data=score_df.head(20), x='final_score', y='feature', ax=ax[1])
ax[1].set_title('Top 20 Final Scores')
plt.tight_layout()

## 11. Agent Logs

In [ ]:
cfg.ensure_dirs()
diagnosis_df.to_csv(cfg.output_dir / 'feature_diagnosis.csv', index=False)
cleaning_df.to_csv(cfg.output_dir / 'feature_cleaning_log.csv', index=False)
score_df.to_csv(cfg.output_dir / 'feature_scores.csv', index=False)
top50_df.to_csv(cfg.output_dir / 'selected_top50.csv', index=False)
metrics_df.to_csv(cfg.output_dir / 'model_metrics.csv', index=False)
print('Saved outputs to', cfg.output_dir)

## 12. Final Conclusion
- Agent Top50 should be compared with random 50 and full features
- All transformations are fitted on train then applied to valid/test
- Leakage checks are enforced chronologically